# Parte 2 — Experimentos no Wiki-RfA

Experimentos de BFS, Dijkstra, DFS e Bellman-Ford, com responsabilidades e revisão registradas na documentação da equipe.

Este notebook lê o CSV tratado produzido por baixar_wikiRfA.ipynb. Execute as células em ordem, em uma sessão nova. Não é necessário executar o notebook de tratamento novamente se o CSV já estiver disponível.

**Pergunta de pesquisa:** Como a transformação dos pesos negativos para permitir o uso de Dijkstra altera os caminhos mínimos e a interpretação dos custos no grafo Wiki-RfA?

Este notebook reúne os experimentos de BFS, Dijkstra e DFS. O benchmark é uma entrega separada.

**Apoio de IA:** ChatGPT (Codex) auxiliou na organização deste notebook e no código de chamada e exportação. O BFS utilizado é a implementação existente do projeto. As células foram criadas sem execução.


## 1. Preparação

Localiza a raiz procurando as pastas src e data no diretório atual e em seus ancestrais. Permite iniciar o notebook pela raiz ou pela pasta do dataset, sem depender das variáveis de outro notebook.


In [15]:
# [IA] ChatGPT (Codex) – localizar o projeto e importar as funções existentes
import json
import math
import sys
from pathlib import Path

raiz_projeto = Path.cwd().resolve()
while not ((raiz_projeto / "src").is_dir() and (raiz_projeto / "data").is_dir()):
    if raiz_projeto == raiz_projeto.parent:
        raise FileNotFoundError("Abra o notebook dentro da pasta do projeto.")
    raiz_projeto = raiz_projeto.parent

if str(raiz_projeto) not in sys.path:
    sys.path.insert(0, str(raiz_projeto))

from src.graphs.io import carregar_dataset
from src.graphs.algorithms import bfs

print("Raiz do projeto:", raiz_projeto)


Raiz do projeto: C:\Users\brend\OneDrive\Documentos\projeto-grafos-g1\projeto-grafos-g1


## 2. Carregamento do grafo

Lê o CSV tratado com a coluna PESO. Preserva direção, laços e arcos paralelos, sem reaplicar o incremento nem alterar o arquivo. BFS ignora os valores dos pesos.


In [16]:
# [IA] ChatGPT (Codex) – carregar o CSV tratado para os experimentos
arquivo_tratado = raiz_projeto / "data" / "dataset_parte2" / "processed" / "wikiRfA_tratado.csv"
grafo_dataset = carregar_dataset(arquivo_tratado, coluna_peso="PESO")
print("Vértices:", len(grafo_dataset))


Vértices: 11376


## 3. BFS em três fontes

Fontes selecionadas a partir dos graus de saída salvos nas métricas: Siva1979 (grau 1.232), 'sed (grau 3, próximo da mediana dos graus positivos) e !---slappdash---! (grau 1). Empates foram resolvidos pelo nome, em ordem lexicográfica do Python, mantendo os identificadores originais.

A quantidade de outros usuários alcançados exclui a própria fonte. A maior distância considera apenas distâncias finitas, incluindo a distância zero da fonte. Para usuários inalcançáveis, a distância é registrada como null no JSON. Predecessor null pode indicar a fonte ou um usuário inalcançável; consulte também a distância.



In [17]:
# [IA] ChatGPT (Codex) – chamar o BFS existente e organizar seus resultados
fontes = ["Siva1979", "'sed", "!---slappdash---!"]
for fonte in fontes:
    if fonte not in grafo_dataset:
        raise ValueError(f"Fonte não encontrada no grafo: {fonte}")

resultados_bfs = []
for fonte in fontes:
    distancias, predecessores = bfs(grafo_dataset, fonte)
    distancias_json = {}
    outros_alcancados = 0
    maior_distancia = 0

    for usuario, distancia in distancias.items():
        if distancia == math.inf:
            distancias_json[usuario] = None
        else:
            distancias_json[usuario] = distancia
            if usuario != fonte:
                outros_alcancados += 1
            if distancia > maior_distancia:
                maior_distancia = distancia

    resultados_bfs.append({
        "fonte": fonte,
        "grau_saida_fonte": len(grafo_dataset[fonte]),
        "outros_usuarios_alcancados": outros_alcancados,
        "maior_distancia_finita": maior_distancia,
        "distancias": distancias_json,
        "predecessores": predecessores,
    })
    print(f"{fonte}: {outros_alcancados} outros usuários alcançados; maior distância = {maior_distancia} arcos.")


Siva1979: 3443 outros usuários alcançados; maior distância = 5 arcos.
'sed: 3444 outros usuários alcançados; maior distância = 6 arcos.
!---slappdash---!: 3444 outros usuários alcançados; maior distância = 7 arcos.


## 4. Exportação dos resultados

Salva os resultados em out/parte2/bfs_fontes.json, identificando a entrada e a unidade das distâncias. Executar novamente sobrescreve apenas esse arquivo. Não altera o CSV ou o JSON das métricas. Este arquivo não substitui o relatório de benchmark parte2_report.json.


In [18]:
# [IA] ChatGPT (Codex) – exportar os resultados das três fontes em JSON
relatorio_bfs = {
    "algoritmo": "BFS",
    "arquivo_entrada": arquivo_tratado.relative_to(raiz_projeto).as_posix(),
    "coluna_peso_carregada": "PESO",
    "pesos_considerados_na_busca": False,
    "unidade_distancia": "arcos",
    "vertices": len(grafo_dataset),
    "resultados": resultados_bfs,
}
pasta_saida = raiz_projeto / "out" / "parte2"
pasta_saida.mkdir(parents=True, exist_ok=True)
arquivo_bfs = pasta_saida / "bfs_fontes.json"
with open(arquivo_bfs, "w", encoding="utf-8") as arquivo:
    json.dump(relatorio_bfs, arquivo, ensure_ascii=False, indent=2, allow_nan=False)

print("Resultados salvos em:", arquivo_bfs)


Resultados salvos em: C:\Users\brend\OneDrive\Documentos\projeto-grafos-g1\projeto-grafos-g1\out\parte2\bfs_fontes.json


## 5. Transformação dos pesos para Dijkstra

Transformação aprovada pela equipe: novo peso = PESO + 1. O menor PESO conferido no CSV foi -1; por isso, o deslocamento de 1 torna os pesos não negativos. A versão transformada fica em grafo_dijkstra, sem modificar grafo_dataset nem o CSV. Direção, ordem dos arcos, laços e arcos paralelos são preservados.

Para um caminho com k arcos, o custo transformado é o custo em PESO mais k. Portanto, esta transformação pode mudar os caminhos mínimos e não deve ser apresentada como preservação dos custos ou dos caminhos originais. PESO já inclui o incremento por repetição e não é a coluna VOTE original.

Esta célula apenas prepara e confere o grafo transformado. Não executa Dijkstra nem salva arquivos.

### Justificativa da transformação

Escolhemos somar 1 ao peso de todos os arcos porque Dijkstra exige pesos não negativos. Com pesos negativos, a justificativa de que uma distância pode ser considerada definitiva deixa de valer. Como o menor PESO encontrado foi −1, somar 1 produz um peso mínimo de zero. Esse é o menor deslocamento constante que elimina os pesos negativos.

A transformação preservou os vértices, os arcos — incluindo paralelos e laços — e suas direções. Também preservou a ordem relativa dos pesos individuais: um arco mais barato continua mais barato que outro. Os pesos anteriores foram mantidos em um grafo separado.

Entretanto, somar 1 a cada arco não afeta todos os caminhos igualmente. Um caminho com mais arcos recebe um acréscimo maior que um caminho com menos arcos, podendo mudar qual caminho tem o menor custo.

Esse comportamento se conecta à nossa pergunta de pesquisa: queremos verificar em quantos pares os caminhos encontrados com os pesos anteriores e transformados coincidem e em quantos diferem. A comparação com Bellman-Ford será feita apenas quando houver um caminho mínimo finito para o destino. Se houver empate de custos, caminhos diferentes podem ser igualmente ótimos; por isso, uma diferença na sequência de vértices não comprova, sozinha, uma mudança causada pela transformação.

**Apoio de IA:** justificativa elaborada pelo grupo e aprovada após revisão de clareza e precisão conceitual por ChatGPT (Codex). O texto descreve a decisão e o plano de comparação; não apresenta resultados dessa comparação.


In [19]:
# [IA] ChatGPT (Codex) – aplicar o deslocamento de pesos aprovado pela equipe em um grafo separado
deslocamento_peso = 1
grafo_dijkstra = {}
arcos_originais = 0
arcos_transformados = 0

for origem, vizinhos in grafo_dataset.items():
    grafo_dijkstra[origem] = []
    arcos_originais += len(vizinhos)
    for destino, peso in vizinhos:
        novo_peso = peso + deslocamento_peso
        assert novo_peso >= 0, "O deslocamento não eliminou todos os pesos negativos."
        grafo_dijkstra[origem].append((destino, novo_peso))
        arcos_transformados += 1

assert len(grafo_dijkstra) == len(grafo_dataset)
assert arcos_transformados == arcos_originais
print("Transformação: novo peso = PESO +", deslocamento_peso)
print("Vértices preservados:", len(grafo_dijkstra))
print("Arcos preservados:", arcos_transformados)
print("Todos os pesos transformados são não negativos.")


Transformação: novo peso = PESO + 1
Vértices preservados: 11376
Arcos preservados: 196614
Todos os pesos transformados são não negativos.


## 6. Dijkstra em cinco pares

Executar após a transformação da seção 5. Os pares foram selecionados a partir dos resultados de BFS, usando as três fontes e destinos nos níveis 1, 5, 2, 6 e 7, respectivamente. Para cada nível escolhido, foi usado o primeiro destino em ordem lexicográfica do Python.

O Dijkstra existente será executado uma vez por origem; os resultados serão consultados para os cinco pares. Os caminhos minimizam a soma de PESO + 1 e podem diferir dos caminhos BFS. O custo registrado refere-se exclusivamente aos pesos transformados, não ao custo mínimo em PESO nem aos votos originais.

Os caminhos registram a sequência de vértices. Havendo arcos paralelos, essa sequência não identifica individualmente cada ocorrência; o custo transformado vem das distâncias devolvidas pelo algoritmo. Não há medição de tempo nesta etapa.


In [20]:
# [IA] ChatGPT (Codex) – chamar Dijkstra e a reconstrução existentes para os cinco pares aprovados
from src.graphs.algorithms import dijkstra, reconstruir_caminho

pares_dijkstra = [
    ("Siva1979", "(aeropagitica)"),
    ("Siva1979", "Dieter Simon"),
    ("'sed", "9cds"),
    ("'sed", "Caltrop"),
    ("!---slappdash---!", "Caltrop"),
]

for origem, destino in pares_dijkstra:
    if origem not in grafo_dijkstra or destino not in grafo_dijkstra:
        raise ValueError(f"Origem ou destino fora do grafo: {origem} -> {destino}")

buscas_dijkstra = {}
resultados_dijkstra = []
for origem, destino in pares_dijkstra:
    if origem not in buscas_dijkstra:
        buscas_dijkstra[origem] = dijkstra(grafo_dijkstra, origem)
    distancias, predecessores = buscas_dijkstra[origem]

    if distancias[destino] == math.inf:
        status = "SEM_CAMINHO"
        caminho = []
        custo = None
        quantidade_arcos = None
    else:
        status = "OK"
        caminho = reconstruir_caminho(distancias, predecessores, destino)
        custo = distancias[destino]
        quantidade_arcos = len(caminho) - 1

    resultados_dijkstra.append({
        "origem": origem,
        "destino": destino,
        "status": status,
        "caminho": caminho,
        "quantidade_arcos": quantidade_arcos,
        "custo_transformado": custo,
    })
    print(f"{origem} -> {destino}: {status}; custo transformado = {custo}; arcos = {quantidade_arcos}")


Siva1979 -> (aeropagitica): OK; custo transformado = 0; arcos = 3
Siva1979 -> Dieter Simon: OK; custo transformado = 6; arcos = 5
'sed -> 9cds: OK; custo transformado = 2; arcos = 2
'sed -> Caltrop: OK; custo transformado = 8; arcos = 7
!---slappdash---! -> Caltrop: OK; custo transformado = 8; arcos = 8


## 7. Exportação dos cinco pares

Salva out/parte2/dijkstra_pares.json, identificando a transformação e os resultados dos cinco pares. Executar novamente sobrescreve esse arquivo. Não modifica o CSV, os grafos, as métricas ou o resultado BFS. Não substitui o relatório de benchmark.


In [21]:
# [IA] ChatGPT (Codex) – exportar caminhos e custos transformados dos cinco pares
relatorio_dijkstra = {
    "algoritmo": "DIJKSTRA",
    "arquivo_entrada": arquivo_tratado.relative_to(raiz_projeto).as_posix(),
    "coluna_peso_base": "PESO",
    "transformacao": "novo_peso = PESO + deslocamento",
    "deslocamento": deslocamento_peso,
    "vertices": len(grafo_dijkstra),
    "buscas_por_origem": len(buscas_dijkstra),
    "resultados": resultados_dijkstra,
}
pasta_saida = raiz_projeto / "out" / "parte2"
pasta_saida.mkdir(parents=True, exist_ok=True)
arquivo_dijkstra = pasta_saida / "dijkstra_pares.json"
with open(arquivo_dijkstra, "w", encoding="utf-8") as arquivo:
    json.dump(relatorio_dijkstra, arquivo, ensure_ascii=False, indent=2, allow_nan=False)

print("Resultados salvos em:", arquivo_dijkstra)


Resultados salvos em: C:\Users\brend\OneDrive\Documentos\projeto-grafos-g1\projeto-grafos-g1\out\parte2\dijkstra_pares.json


## 8. DFS em três fontes — André

Executar após a preparação e o carregamento do grafo (seções 1 e 2). Reutiliza as três fontes já registradas no BFS: `Siva1979`, `'sed` e `!---slappdash---!`. Usa o grafo com PESO, sem transformação adicional; o DFS ignora os pesos e preserva a ordem dos vizinhos do carregador. A busca fica restrita à fonte escolhida, com `cobrir_todos=False`.

São registrados ordem de descoberta, tempos, predecessores, contagens de classificação e indicação de ciclos. Os tempos são contadores de eventos da DFS, não medições de desempenho. Não são calculados caminhos mínimos nem camadas de distância nesta seção. A classificação exportada é uma contagem por tipo; os registros individuais de cada arco não são exportados.

**Apoio de IA:** ChatGPT (Codex) auxiliou nas chamadas da função existente, exportação e conferências técnicas. André deve conferir os resultados e elaborar a interpretação autoral. O código do algoritmo não foi alterado nesta etapa.


In [22]:
# [IA] ChatGPT (Codex) – chamar o DFS existente em três fontes e organizar os retornos
from src.graphs.algorithms import dfs

# Reutiliza as fontes já registradas nos experimentos BFS; André deve confirmar essa seleção.
fontes_dfs = ["Siva1979", "'sed", "!---slappdash---!"]
resultados_dfs = []
for fonte in fontes_dfs:
    if fonte not in grafo_dataset:
        raise ValueError(f"Fonte não encontrada no grafo: {fonte}")
    descoberta, termino, predecessores, classificacao, tem_ciclo = dfs(
        grafo_dataset, fonte, cobrir_todos=False
    )
    contagens = {tipo: 0 for tipo in ("arvore", "retorno", "avanco", "cruzamento")}
    for tipo in classificacao.values():
        contagens[tipo] += 1

    # Conferência dos retornos e da preservação de cada ocorrência de arco.
    assert set(descoberta) == set(termino) == set(predecessores)
    assert predecessores[fonte] is None
    assert all(descoberta[v] < termino[v] for v in descoberta)
    assert len(classificacao) == sum(len(grafo_dataset[v]) for v in descoberta)
    assert contagens["arvore"] == len(descoberta) - 1
    assert tem_ciclo == (contagens["retorno"] > 0)
    assert all(destino in descoberta for origem in descoberta for destino, _ in grafo_dataset[origem])

    resultados_dfs.append({
        "fonte": fonte,
        "vertices_alcancados_incluindo_fonte": len(descoberta),
        "outros_usuarios_alcancados": len(descoberta) - 1,
        "ordem_descoberta": list(descoberta),
        "tempos_descoberta": descoberta,
        "tempos_termino": termino,
        "predecessores": predecessores,
        "arcos_classificados": len(classificacao),
        "classificacao_arcos": contagens,
        "tem_ciclo": tem_ciclo,
    })
    print(f"{fonte}: {len(descoberta)} vértices; {len(classificacao)} arcos classificados; ciclo = {tem_ciclo}.")


Siva1979: 3444 vértices; 125455 arcos classificados; ciclo = True.
'sed: 3445 vértices; 125458 arcos classificados; ciclo = True.
!---slappdash---!: 3445 vértices; 125456 arcos classificados; ciclo = True.


## 9. Exportação do DFS

Salva `out/parte2/dfs_fontes.json`. Executar novamente sobrescreve apenas esse arquivo. Não modifica CSV, métricas ou resultados de BFS e Dijkstra. A saída não substitui o benchmark `parte2_report.json` nem a análise autoral no PDF.


In [23]:
# [IA] ChatGPT (Codex) – exportar os retornos do DFS existente, sem interpretação dos resultados
relatorio_dfs = {
    "algoritmo": "DFS",
    "arquivo_entrada": arquivo_tratado.relative_to(raiz_projeto).as_posix(),
    "coluna_peso_carregada": "PESO",
    "pesos_considerados_na_busca": False,
    "cobrir_todos": False,
    "ordem_vizinhos": "ordem dos arcos no CSV tratado, preservada pelo carregador",
    "identidade_arco": "(origem, destino, indice_na_adjacencia), com indice iniciando em zero",
    "vertices": len(grafo_dataset),
    "resultados": resultados_dfs,
}
pasta_saida_dfs = raiz_projeto / "out" / "parte2"
pasta_saida_dfs.mkdir(parents=True, exist_ok=True)
arquivo_dfs = pasta_saida_dfs / "dfs_fontes.json"
with open(arquivo_dfs, "w", encoding="utf-8") as saida:
    json.dump(relatorio_dfs, saida, ensure_ascii=False, indent=2, allow_nan=False)
print("Resultados DFS salvos em:", arquivo_dfs)


Resultados DFS salvos em: C:\Users\brend\OneDrive\Documentos\projeto-grafos-g1\projeto-grafos-g1\out\parte2\dfs_fontes.json


## Interpretação dos resultados do DFS 

O DFS alcançou 3.444 vértices a partir de Siva1979 e 3.445 a partir de 'sed e !---slappdash---!, incluindo cada fonte. Nenhuma delas alcançou todos os 11.376 vértices do grafo. As quantidades próximas não garantem que os conjuntos visitados sejam iguais.

As três execuções encontraram arcos de retorno, indicando ciclos dirigidos. Isso não comprova ciclos negativos, pois o DFS ignora os pesos.

A quantidade de arcos de árvore correspondeu ao número de vértices alcançados menos um. Os resultados descrevem alcançabilidade e ciclos, sem determinar caminhos ou custos mínimos.

Uso de IA: Codex me auxiliou na gramática do meu texto de análise e posteriormente sintetizou a minha frase para deixar ela mais direta e concisa.

## 10. Bellman-Ford 

Executar após a preparação e o carregamento (seções 1 e 2). Os casos abaixo são sugestões de Codex extraídas do CSV tratado.

- **Peso negativo sem ciclo negativo:** usuários `Malleus Fatuorum` e `BDD`; fonte `Malleus Fatuorum`.
- **Com ciclo negativo:** usuários `Rcsprinter123` e `Jasper Deng`; fonte `Rcsprinter123`.

Em ambos, o filtro mantém todas as linhas em que SOURCE e TARGET pertencem ao conjunto de usuários do caso: são subgrafos induzidos. Usa a coluna PESO já calculada, sem a transformação de Dijkstra e sem inventar votos ou pesos. Os números de linha identificam o CSV tratado exportado pelo projeto, com o cabeçalho na linha 1; não são índices do CSV bruto.

São exemplos mínimos para conferir os dois comportamentos. Não constituem amostras representativas da rede completa e não substituem os três tamanhos do benchmark.

As chamadas usam o Bellman-Ford existente, sem alterar sua implementação. A permissão para dispensar novos marcadores foi informada por André; o apoio de IA continua registrado neste texto e no log.


In [24]:
import csv
from src.graphs.algorithms import bellman_ford, reconstruir_caminho
from src.graphs.graph import adicionar_arco

# [IA] ChatGPT (Codex) – Sugestões de casos mínimos extraídos do CSV
casos_bellman = [
    {
        "id": "negativo_sem_ciclo",
        "usuarios": ["Malleus Fatuorum", "BDD"],
        "fonte": "Malleus Fatuorum",
        "espera_ciclo_negativo": False,
    },
    {
        "id": "ciclo_negativo",
        "usuarios": ["Rcsprinter123", "Jasper Deng"],
        "fonte": "Rcsprinter123",
        "espera_ciclo_negativo": True,
    },
]

with open(arquivo_tratado, newline="", encoding="utf-8-sig") as entrada:
    registros_bellman = list(csv.DictReader(entrada))

resultados_bellman = []
for caso in casos_bellman:
    usuarios_caso = set(caso["usuarios"])
    grafo_caso = {usuario: [] for usuario in caso["usuarios"]}
    arcos_caso = []
    # [IA] ChatGPT (Codex) – Recorte induzido: mantém todas as linhas cujos dois extremos pertencem ao par.
    # [IA] ChatGPT (Codex) – Não altera pesos, direção, laços ou ocorrências paralelas.
    for indice, registro in enumerate(registros_bellman):
        origem, destino = registro["SOURCE"], registro["TARGET"]
        if origem in usuarios_caso and destino in usuarios_caso:
            peso = int(registro["PESO"])
            indice_adjacencia = len(grafo_caso[origem])
            adicionar_arco(grafo_caso, origem, destino, peso)
            arcos_caso.append({
                "indice_registro_csv_base_zero": indice,
                "linha_csv_com_cabecalho": indice + 2,
                "origem": origem,
                "destino": destino,
                "indice_adjacencia": indice_adjacencia,
                "VOTE": int(registro["VOTE"]),
                "RESULT": int(registro["RESULT"]),
                "PESO": peso,
            })

    assert caso["fonte"] in grafo_caso
    assert any(arco["PESO"] < 0 for arco in arcos_caso)
    distancias, predecessores, ciclo = bellman_ford(grafo_caso, caso["fonte"])
    assert (ciclo is not None) == caso["espera_ciclo_negativo"]

    resultado = {
        "id": caso["id"],
        "fonte": caso["fonte"],
        "usuarios": caso["usuarios"],
        "vertices": len(grafo_caso),
        "arcos": arcos_caso,
        "tem_ciclo_negativo": ciclo is not None,
    }
    if ciclo is None:
        resultado["status"] = "OK"
        resultado["distancias"] = {
            v: None if d == math.inf else d for v, d in distancias.items()
        }
        resultado["predecessores"] = predecessores
        resultado["caminhos"] = {
            v: [] if distancias[v] == math.inf else reconstruir_caminho(distancias, predecessores, v)
            for v in grafo_caso
        }
        assert all(d != math.inf for d in distancias.values()), "Os usuários do exemplo devem ser alcançáveis."
        assert distancias["Malleus Fatuorum"] == 0
        assert distancias["BDD"] == -1
        print(caso["id"], "fonte:", caso["fonte"], "distâncias:", resultado["distancias"])
    else:
        resultado["status"] = "CICLO_NEGATIVO"
        assert ciclo[0] == ciclo[-1]
        arcos_do_ciclo = []
        for origem, destino in zip(ciclo, ciclo[1:]):
            candidatos = [a for a in arcos_caso if a["origem"] == origem and a["destino"] == destino]
            # [IA] ChatGPT (Codex) – Os exemplos propostos têm um único arco em cada direção do ciclo.
            # [IA] ChatGPT (Codex) – Se a seleção mudar e houver paralelos, não escolher uma ocorrência silenciosamente.
            if len(candidatos) != 1:
                raise ValueError("Ciclo com ocorrências paralelas: é necessário identificar o arco utilizado.")
            arcos_do_ciclo.append(candidatos[0])
        custo_ciclo = sum(a["PESO"] for a in arcos_do_ciclo)
        assert custo_ciclo < 0
        resultado["ciclo"] = ciclo
        resultado["arcos_ciclo"] = arcos_do_ciclo
        resultado["custo_ciclo"] = custo_ciclo
        resultado["distancias"] = None
        resultado["observacao_distancias"] = "Valores provisórios da execução com ciclo negativo não são apresentados como custos mínimos finitos."
        print(caso["id"], "fonte:", caso["fonte"], "ciclo:", ciclo, "custo:", custo_ciclo)
    resultados_bellman.append(resultado)


negativo_sem_ciclo fonte: Malleus Fatuorum distâncias: {'Malleus Fatuorum': 0, 'BDD': -1}
ciclo_negativo fonte: Rcsprinter123 ciclo: ['Jasper Deng', 'Rcsprinter123', 'Jasper Deng'] custo: -2


## 11. Exportação dos casos de Bellman-Ford

Salva `out/parte2/bellman_ford_casos.json`, com fontes, usuários, linhas do CSV tratado, pesos e resultados. No caso sem ciclo negativo, exporta distâncias, predecessores e caminhos. No caso com ciclo negativo, exporta a sequência, os arcos identificados e o custo do ciclo. As distâncias provisórias desse segundo caso não são exportadas como mínimos finitos.

Executar novamente sobrescreve apenas essa saída. Não modifica os dados nem os resultados dos demais algoritmos.


In [25]:
# [IA] ChatGPT (Codex) – Auxílio na lógica do código
relatorio_bellman = {
    "algoritmo": "BELLMAN_FORD",
    "arquivo_entrada": arquivo_tratado.relative_to(raiz_projeto).as_posix(),
    "sha256_entrada": __import__("hashlib").sha256(arquivo_tratado.read_bytes()).hexdigest(),
    "coluna_peso": "PESO",
    "regra_recorte": "subgrafo induzido pelos usuários informados; todas as ocorrências entre eles são preservadas",
    "selecao": "exemplos mínimos sugeridos por Codex para conferência e justificativa autoral de André",
    "benchmark": False,
    "resultados": resultados_bellman,
}
arquivo_bellman = raiz_projeto / "out" / "parte2" / "bellman_ford_casos.json"
arquivo_bellman.parent.mkdir(parents=True, exist_ok=True)
with open(arquivo_bellman, "w", encoding="utf-8") as saida:
    json.dump(relatorio_bellman, saida, ensure_ascii=False, indent=2, allow_nan=False)
print("Casos Bellman-Ford salvos em:", arquivo_bellman)


Casos Bellman-Ford salvos em: C:\Users\brend\OneDrive\Documentos\projeto-grafos-g1\projeto-grafos-g1\out\parte2\bellman_ford_casos.json


## Justificativas e interpretação dos casos de Bellman-Ford 

Foram usados dois recortes pequenos do Wiki-RfA, com o objetivo de verificar os comportamentos do Bellman-Ford com pesos negativos. Cada recorte preserva todos os arcos entre os dois usuários selecionados e seus valores de PESO, facilitando a conferência dos resultados.

No primeiro caso, com fonte em Malleus Fatuorum, há um arco até BDD com peso −1 e nenhum caminho de volta. O algoritmo retornou distância −1 até BDD e não identificou ciclo negativo. No segundo, Rcsprinter123 e Jasper Deng possuem um arco em cada direção, ambos com peso −1. A soma dos pesos do ciclo é −2, compatível com a detecção de ciclo negativo pelo algoritmo.

Os exemplos permitem conferir a diferença entre encontrar um caminho de custo negativo e detectar um ciclo negativo alcançável pela fonte. Como os recortes possuem apenas dois usuários, seus resultados não descrevem o comportamento de toda a rede nem permitem avaliar o desempenho do algoritmo em grafos maiores.

Uso de IA: Codex me auxiliou na gramática do meu texto de análise e posteriormente sintetizou a minha frase para deixar ela mais direta e concisa.


## 12. Preparação do benchmark de BFS e Dijkstra 

Estas seções dependem apenas da preparação e do carregamento das seções 1 e 2 e do arquivo src/benchmark.py atualizado. Não dependem de executar DFS ou Bellman-Ford. Os recortes são preparados antes da medição, com PESO já calculado e semente 42. As fontes são escolhidas no recorte de 10% e mantidas nos três tamanhos.

A execução abaixo apenas prepara recortes e fontes; não mede algoritmos nem salva arquivos.


In [26]:
# [IA] ChatGPT (Codex) – integrar a preparação aprovada dos recortes e fontes
from src.benchmark import criar_recortes, selecionar_fontes, medir_algoritmo
from src.graphs.algorithms import bfs, dijkstra

semente_benchmark = 42
repeticoes_benchmark = 5
deslocamento_benchmark = 1
recortes_benchmark = criar_recortes(grafo_dataset, semente=semente_benchmark)
fontes_benchmark = selecionar_fontes(recortes_benchmark)

print("Fontes do benchmark:", fontes_benchmark)
for recorte in recortes_benchmark:
    print(recorte["percentual"], "%:", recorte["vertices"], "vértices e", recorte["arcos"], "arcos")


Fontes do benchmark: ['Siva1979', 'Gwynand', 'John Carter']
10 %: 5636 vértices e 19661 arcos
50 %: 9187 vértices e 98307 arcos
100 %: 11376 vértices e 196614 arcos


## 13. Alcance das fontes fora da medição

Executa BFS uma vez por fonte e tamanho apenas para contar os outros usuários alcançados, excluindo a própria fonte. Essas chamadas não são aquecimento nem repetições medidas. Como a transformação de pesos preserva os arcos, o alcance também vale para a versão usada por Dijkstra.


In [27]:
# [IA] ChatGPT (Codex) – registrar alcance sem incluir esse cálculo no tempo medido
alcances_benchmark = {}
for recorte in recortes_benchmark:
    percentual = recorte["percentual"]
    for fonte in fontes_benchmark:
        distancias_alcance, predecessores_alcance = bfs(recorte["grafo"], fonte)
        outros_alcancados = 0
        for usuario, distancia in distancias_alcance.items():
            if usuario != fonte and distancia != math.inf:
                outros_alcancados += 1
        alcances_benchmark[(percentual, fonte)] = outros_alcancados
        print(percentual, "% |", fonte, "| outros usuários alcançados:", outros_alcancados)
        del distancias_alcance, predecessores_alcance


10 % | Siva1979 | outros usuários alcançados: 2661
10 % | Gwynand | outros usuários alcançados: 2661
10 % | John Carter | outros usuários alcançados: 2661
50 % | Siva1979 | outros usuários alcançados: 3360
50 % | Gwynand | outros usuários alcançados: 3360
50 % | John Carter | outros usuários alcançados: 3360
100 % | Siva1979 | outros usuários alcançados: 3443
100 % | Gwynand | outros usuários alcançados: 3443
100 % | John Carter | outros usuários alcançados: 3443


## 14. Medições de BFS e Dijkstra

São 18 configurações: 2 algoritmos × 3 fontes × 3 tamanhos. Cada configuração tem um aquecimento fora do cronômetro e cinco repetições medidas. São 90 chamadas medidas e 18 aquecimentos, além das nove chamadas da seção de alcance.

Apenas a chamada do algoritmo entra no tempo. A criação do grafo de Dijkstra, a contagem de alcance e a organização dos resultados ficam fora. BFS usa PESO e Dijkstra usa PESO + 1 nos mesmos arcos de cada recorte. A tarefa é uma busca completa por fonte; não são os cinco pares dos casos obrigatórios.

Os resultados ficam na variável medicoes_bfs_dijkstra, sem salvar arquivos nesta etapa. Ela é reiniciada ao executar a célula novamente. O relatório final ainda depende de identificação do ambiente, exportação e consolidação com as medições de DFS e Bellman-Ford.


In [28]:
# [IA] ChatGPT (Codex) – organizar medições de BFS e Dijkstra pelo procedimento comum aprovado
medicoes_bfs_dijkstra = []
for recorte in recortes_benchmark:
    percentual = recorte["percentual"]
    grafo_base = recorte["grafo"]

    # Preparação dos pesos fora de medir_algoritmo.
    grafo_dijkstra_benchmark = {}
    for origem, vizinhos in grafo_base.items():
        grafo_dijkstra_benchmark[origem] = []
        for destino, peso in vizinhos:
            novo_peso = peso + deslocamento_benchmark
            assert novo_peso >= 0, "Há peso negativo na versão de Dijkstra."
            grafo_dijkstra_benchmark[origem].append((destino, novo_peso))

    tarefas = [
        ("BFS", bfs, grafo_base, "PESO", 0),
        ("DIJKSTRA", dijkstra, grafo_dijkstra_benchmark, "PESO + 1", deslocamento_benchmark),
    ]
    for nome, algoritmo, grafo_medicao, versao_pesos, deslocamento in tarefas:
        for fonte in fontes_benchmark:
            medicao = medir_algoritmo(algoritmo, grafo_medicao, fonte, repeticoes=repeticoes_benchmark)
            medicao.update({
                "algoritmo": nome,
                "tarefa": "busca_completa_por_fonte",
                "fonte": fonte,
                "percentual": percentual,
                "vertices": recorte["vertices"],
                "arcos": recorte["arcos"],
                "grau_saida_fonte": len(grafo_base[fonte]),
                "outros_usuarios_alcancados": alcances_benchmark[(percentual, fonte)],
                "versao_pesos": versao_pesos,
                "deslocamento": deslocamento,
                "semente": semente_benchmark,
            })
            medicoes_bfs_dijkstra.append(medicao)
            print(f"{nome} | {percentual}% | {fonte} | média: {medicao['media']:.6f} segundos")


BFS | 10% | Siva1979 | média: 0.022986 segundos
BFS | 10% | Gwynand | média: 0.026771 segundos
BFS | 10% | John Carter | média: 0.019035 segundos
DIJKSTRA | 10% | Siva1979 | média: 0.020963 segundos
DIJKSTRA | 10% | Gwynand | média: 0.017418 segundos
DIJKSTRA | 10% | John Carter | média: 0.014280 segundos
BFS | 50% | Siva1979 | média: 0.094029 segundos
BFS | 50% | Gwynand | média: 0.115139 segundos
BFS | 50% | John Carter | média: 0.108647 segundos
DIJKSTRA | 50% | Siva1979 | média: 0.075709 segundos
DIJKSTRA | 50% | Gwynand | média: 0.075721 segundos
DIJKSTRA | 50% | John Carter | média: 0.069701 segundos
BFS | 100% | Siva1979 | média: 0.146239 segundos
BFS | 100% | Gwynand | média: 0.168302 segundos
BFS | 100% | John Carter | média: 0.158078 segundos
DIJKSTRA | 100% | Siva1979 | média: 0.122557 segundos
DIJKSTRA | 100% | Gwynand | média: 0.106724 segundos
DIJKSTRA | 100% | John Carter | média: 0.099202 segundos


## 15. Conferência e exportação do benchmark de BFS e Dijkstra

Executar na mesma sessão das medições, após a seção 14. Confere as 18 configurações, o aquecimento e pelo menos cinco tempos por configuração. Registra o ambiente da sessão; use a mesma máquina em que as medições foram feitas. Não executa os algoritmos novamente.

O arquivo out/parte2/benchmark_bfs_dijkstra.json guarda esta contribuição, sem substituir parte2_report.json. Executar novamente sobrescreve apenas esse arquivo. Os caminhos são relativos, sem nome de usuário ou computador. Se psutil não estiver instalado, a memória fica como null; não há instalação automática de dependências. Informações indisponíveis devem ser completadas antes do relatório final.


In [29]:
# [IA] ChatGPT (Codex) – conferir configurações e exportar medições e ambiente sem dados pessoais
import platform

configuracoes_esperadas = set()
for algoritmo in ["BFS", "DIJKSTRA"]:
    for percentual in [10, 50, 100]:
        for fonte in fontes_benchmark:
            configuracoes_esperadas.add((algoritmo, percentual, fonte))

assert len(medicoes_bfs_dijkstra) == 18, "São esperadas 18 configurações."
configuracoes_encontradas = set()
for medicao in medicoes_bfs_dijkstra:
    configuracoes_encontradas.add((medicao["algoritmo"], medicao["percentual"], medicao["fonte"]))
    assert medicao["aquecimentos"] == 1
    assert medicao["repeticoes"] >= 5
    assert len(medicao["tempos"]) == medicao["repeticoes"]
    assert medicao["unidade_tempo"] == "segundos"
    for tempo in medicao["tempos"]:
        assert math.isfinite(tempo) and tempo >= 0, "Tempo inválido."
assert configuracoes_encontradas == configuracoes_esperadas

processador = platform.processor() or None
if platform.system() == "Windows":
    import winreg
    try:
        with winreg.OpenKey(winreg.HKEY_LOCAL_MACHINE, r"HARDWARE\DESCRIPTION\System\CentralProcessor\0") as chave:
            processador = winreg.QueryValueEx(chave, "ProcessorNameString")[0].strip()
    except OSError:
        pass  # Mantém a informação obtida por platform.processor().

try:
    import psutil
    memoria_total_bytes = psutil.virtual_memory().total
except ImportError:
    memoria_total_bytes = None

ambiente_benchmark = {
    "python": platform.python_version(),
    "sistema": platform.system(),
    "versao_sistema": platform.version(),
    "processador": processador,
    "memoria_total_bytes": memoria_total_bytes,
}
relatorio_bfs_dijkstra = {
    "arquivo_entrada": arquivo_tratado.relative_to(raiz_projeto).as_posix(),
    "ambiente": ambiente_benchmark,
    "semente": semente_benchmark,
    "regra_recorte": "prefixos de 10%, 50% e 100% das ocorrências embaralhadas uma vez; PESO preservado",
    "regra_fontes": "no recorte de 10%, grau de saída >= 3; ordem por grau decrescente e nome crescente; posições 0, n // 4, n // 2",
    "fontes": fontes_benchmark,
    "alcance_inclui_fonte": False,
    "intervalo_medido": "somente a chamada do algoritmo; busca completa por fonte",
    "medicoes": medicoes_bfs_dijkstra,
}
arquivo_benchmark = raiz_projeto / "out" / "parte2" / "benchmark_bfs_dijkstra.json"
arquivo_benchmark.parent.mkdir(parents=True, exist_ok=True)
with open(arquivo_benchmark, "w", encoding="utf-8") as arquivo:
    json.dump(relatorio_bfs_dijkstra, arquivo, ensure_ascii=False, indent=2, allow_nan=False)

print("Configurações conferidas:", len(medicoes_bfs_dijkstra))
print("Benchmark salvo em:", arquivo_benchmark.relative_to(raiz_projeto))
if processador is None or memoria_total_bytes is None:
    print("Há informações de hardware indisponíveis; complete-as antes do relatório final.")


Configurações conferidas: 18
Benchmark salvo em: out\parte2\benchmark_bfs_dijkstra.json
